# 2. Function 1 baseline
Suggest a maximisation candidate using **only the original initial observations** from `initial_inputs.npy` and `initial_outputs.npy`. Saved evaluations in `all_observations.npz` are excluded from this baseline. The evaluator is external; do not execute the recording cell until a real scalar output is received.

In [1]:
from pathlib import Path
from bayesian_bbo import BayesianOptimizer, OptimisationConfig, append_observation, format_submission, load_observations, remaining_budget
import numpy as np
from bayesian_bbo.surrogate import fit_surrogate
from sklearn.gaussian_process.kernels import RBF

import plotly.graph_objects as go


def create_grid_coordinates(slice_dimensions, grid_size=101):
    grid_values = np.linspace(0, 1, grid_size, endpoint=False)
    return np.meshgrid(
        *(grid_values for _ in slice_dimensions), indexing='ij'
    )


def plot_optimization_surface(
    history, model, slice_dimensions, beta=1.96, suggested_point=None,
):
    if len(slice_dimensions) != 2:
        raise ValueError('A 3D surface requires two varying input dimensions.')

    horizontal, vertical = slice_dimensions
    grid_coordinates = create_grid_coordinates(slice_dimensions)
    grid_horizontal, grid_vertical = grid_coordinates
    x_grid = np.tile(history.best_input, (grid_horizontal.size, 1))
    for dimension, coordinates in zip(slice_dimensions, grid_coordinates):
        x_grid[:, dimension] = coordinates.ravel()
    post_mean, post_std = model.predict(x_grid, return_std=True)
    assert x_grid.shape[1] == history.dimension
    mean_surface = np.asarray(post_mean).reshape(grid_horizontal.shape)
    std_surface = np.asarray(post_std).reshape(grid_horizontal.shape)

    fig = go.Figure()
    fig.add_trace(go.Surface(
        x=grid_horizontal, y=grid_vertical, z=mean_surface,
        colorscale='Viridis', opacity=0.85,
        colorbar=dict(title='Predicted output'),
        name='GP posterior mean', showlegend=True,
        hovertemplate=(
            f'x{horizontal + 1}: %{{x:.4f}}<br>'
            f'x{vertical + 1}: %{{y:.4f}}<br>'
            'Mean: %{z:.4f}<extra></extra>'
        ),
    ))
    for sign, name in ((-1, 'Lower uncertainty bound'), (1, 'Upper uncertainty bound')):
        fig.add_trace(go.Surface(
            x=grid_horizontal, y=grid_vertical,
            z=mean_surface + sign * beta * std_surface,
            colorscale=[[0, 'gray'], [1, 'gray']],
            opacity=0.15, showscale=False, hoverinfo='skip',
            name=f'Mean +/- {beta} standard deviations',
            legendgroup='uncertainty', showlegend=sign == 1,
        ))

    fixed_dimensions = [
        dimension for dimension in range(history.dimension)
        if dimension not in slice_dimensions
    ]
    on_slice = np.all(
        np.isclose(
            history.inputs[:, fixed_dimensions],
            history.best_input[fixed_dimensions],
            rtol=0, atol=1e-8,
        ),
        axis=1,
    )
    fig.add_trace(go.Scatter3d(
        x=history.inputs[on_slice, horizontal],
        y=history.inputs[on_slice, vertical],
        z=history.outputs[on_slice],
        mode='markers', name='Queries on slice',
        marker=dict(color='red', symbol='circle', size=5),
        hovertemplate=(
            f'x{horizontal + 1}: %{{x:.4f}}<br>'
            f'x{vertical + 1}: %{{y:.4f}}<br>'
            'Measured output: %{z:.4f}<extra></extra>'
        ),
    ))
    if suggested_point is not None:
        suggested_mean = model.predict(np.asarray(suggested_point).reshape(1, -1))[0]
        fig.add_trace(go.Scatter3d(
            x=[suggested_point[horizontal]],
            y=[suggested_point[vertical]],
            z=[suggested_mean],
            mode='markers', name='Suggested candidate',
            marker=dict(color='blue', symbol='cross', size=8),
            hovertemplate=(
                f'x{horizontal + 1}: %{{x:.4f}}<br>'
                f'x{vertical + 1}: %{{y:.4f}}<br>'
                'Predicted mean: %{z:.4f}<extra></extra>'
            ),
        ))
    fig.update_layout(
        title=f'Function {function_id}: GP posterior slice',
        template='plotly_white', height=500, autosize=True,
        margin=dict(l=0, r=0, b=0, t=80),
        legend=dict(orientation='h', x=0, y=1.05),
        scene=dict(
            xaxis=dict(title=f'x{horizontal + 1}', range=[0, 1]),
            yaxis=dict(title=f'x{vertical + 1}', range=[0, 1]),
            zaxis=dict(title='Output'),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.1)),
            aspectmode='cube',
        ),
    )
    fig.show(renderer='plotly_mimetype')
    return fig


root = Path('../../data')
kappa = 1.96

In [2]:
from bayesian_bbo import CONFIG

In [3]:
from bayesian_bbo.data import FUNCTION_SPECS, Observations


def load_initial_observations(root, function_id):
    directory = Path(root) / f'function_{function_id}'
    return Observations(
        np.load(directory / 'initial_inputs.npy', allow_pickle=False),
        np.load(directory / 'initial_outputs.npy', allow_pickle=False),
        FUNCTION_SPECS[function_id],
    )

Context: evaluation of first set of 

In [4]:
function_id = 1
observations = load_initial_observations(root, function_id)
num_queries = 100
kernel = RBF(length_scale=0.1, length_scale_bounds="fixed")

optimizer = BayesianOptimizer(
    observations,
    OptimisationConfig(
        budget=remaining_budget(observations, function_id, num_queries),
        seed=CONFIG["random_state"],
        acquisition="ucb",
        kappa=kappa,
        kernel=kernel,
    ),
)
point, mean, std, acquisition_score = optimizer.suggest_with_prediction()
print(f'Function {function_id}: submit {format_submission(point)}')
print(f'Predicted mean: {mean:.4f}, posterior std: {std:.4f}, UCB: {acquisition_score:.4f}')

history = optimizer.history
slice_dimensions = tuple(range(min(2, history.dimension)))
model = fit_surrogate(history, kernel=kernel)
fig = plot_optimization_surface(
    history, model, slice_dimensions,
    beta=kappa, suggested_point=point,
)

Function 1: submit 0.835456-0.755330
Predicted mean: 0.0012, posterior std: 0.0008, UCB: 0.0028


In [5]:
function_id = 2
observations = load_initial_observations(root, function_id)
num_queries = 100
kernel = RBF(length_scale=0.1, length_scale_bounds="fixed")

optimizer = BayesianOptimizer(
    observations,
    OptimisationConfig(
        budget=remaining_budget(observations, function_id, num_queries),
        seed=CONFIG["random_state"],
        acquisition="ucb",
        kappa=kappa,
        kernel=kernel,
    ),
)
point, mean, std, acquisition_score = optimizer.suggest_with_prediction()
print(f'Function {function_id}: submit {format_submission(point)}')
print(f'Predicted mean: {mean:.4f}, posterior std: {std:.4f}, UCB: {acquisition_score:.4f}')

history = optimizer.history
slice_dimensions = tuple(range(min(2, history.dimension)))
model = fit_surrogate(history, kernel=kernel)
fig = plot_optimization_surface(
    history, model, slice_dimensions,
    beta=kappa, suggested_point=point,
)

Function 2: submit 0.802058-0.932779
Predicted mean: 0.5236, posterior std: 0.1709, UCB: 0.8585


In [6]:
num_queries = 100
rbf_lengthscale = 0.1

suggested_points = {}
optimizers_by_function = {}
expected_values = {}
standard_deviations = {}
acquisition_scores = {}

for function_id in range(1, 9):
    observations = load_initial_observations(root, function_id)
    budget = remaining_budget(observations, function_id, num_queries)
    if budget == 0:
        print(f'Function {function_id}: query budget exhausted.')
        continue

    kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds="fixed")
    optimizer = BayesianOptimizer(
        observations,
        OptimisationConfig(
            budget=budget,
            seed=CONFIG["random_state"],
            acquisition="ucb",
            kappa=kappa,
            kernel=kernel,
        ),
    )

    point, mean, std, score = optimizer.suggest_with_prediction()
    suggested_points[function_id] = point
    optimizers_by_function[function_id] = optimizer
    expected_values[function_id] = mean
    standard_deviations[function_id] = std
    acquisition_scores[function_id] = score

    mean_magnitude = abs(mean)
    ratio = std / max(mean_magnitude, 1e-12)
    nearest_distance = np.linalg.norm(observations.inputs - point, axis=1).min()

    print(f'Function {function_id}: submit {format_submission(point)}')
    print(
        f'kappa used for selection: {kappa:.2f}, '
        f'mean: {mean:.4f}, '
        f'raw posterior std: {std:.4f}, '
        f'std/|mean|: {ratio:.4f}, '
        f'UCB (includes kappa): {score:.4f}, '
        f'nearest observed distance: {nearest_distance:.4f}'
    )

Function 1: submit 0.835456-0.755330
kappa used for selection: 1.96, mean: 0.0012, raw posterior std: 0.0008, std/|mean|: 0.7085, UCB (includes kappa): 0.0028, nearest observed distance: 0.1068
Function 2: submit 0.802058-0.932779
kappa used for selection: 1.96, mean: 0.5236, raw posterior std: 0.1709, std/|mean|: 0.3264, UCB (includes kappa): 0.8585, nearest observed distance: 0.0996
Function 3: submit 0.194638-0.363906-0.466721
kappa used for selection: 1.96, mean: -0.0726, raw posterior std: 0.0761, std/|mean|: 1.0485, UCB (includes kappa): 0.0766, nearest observed distance: 0.1422
Function 4: submit 0.528998-0.424017-0.506366-0.277697
kappa used for selection: 1.96, mean: -8.9734, raw posterior std: 5.5308, std/|mean|: 0.6164, UCB (includes kappa): 1.8670, nearest observed distance: 0.0985
Function 5: submit 0.296670-0.806944-0.906495-0.936907
kappa used for selection: 1.96, mean: 689.5015, raw posterior std: 200.3223, std/|mean|: 0.2905, UCB (includes kappa): 1082.1331, nearest ob

Function 8: submit 0.133138-0.573640-0.420506-0.497998-0.832146-0.580438-0.155169-0.376735
kappa used for selection: 1.96, mean: 8.0821, raw posterior std: 0.9324, std/|mean|: 0.1154, UCB (includes kappa): 9.9096, nearest observed distance: 0.1869


In [7]:
import json
from bayesian_bbo.data import save_predictions

function_ids = range(1, 9)
best_outputs = [
    float(np.max(np.load(root / f'function_{function_id}' / 'initial_outputs.npy')))
    for function_id in function_ids
]
week_directory = root / 'weekly_updates' / 'week_2'
inputs_path = week_directory / 'inputs.json'
if inputs_path.exists():
    assert json.loads(inputs_path.read_text()) == [
        np.round(suggested_points[function_id], 6).tolist() for function_id in function_ids
    ], 'Regenerated candidates differ from the historical submitted inputs.'
saved_prediction_paths = save_predictions(
    [expected_values[function_id] for function_id in function_ids],
    [suggested_points[function_id] for function_id in function_ids],
    [standard_deviations[function_id] for function_id in function_ids],
    best_outputs,
    week_directory,
    acquisition_scores=[acquisition_scores[function_id] for function_id in function_ids],
    acquisition_settings=[
        optimizers_by_function[function_id].acquisition_settings
        for function_id in function_ids
    ],
    overwrite=True,
)
saved_prediction_paths

Saved and verified ../../data/weekly_updates/week_2/inputs.json
Saved and verified ../../data/weekly_updates/week_2/predictions.json


{'inputs.json': PosixPath('../../data/weekly_updates/week_2/inputs.json'),
 'predictions.json': PosixPath('../../data/weekly_updates/week_2/predictions.json')}

# Analysis

**What was the main principle or heuristic you used to decide on each query point?**

For the first round, I used the UCB acquisition function with the same `kappa = 1.96` for all functions. This balances the predicted output against uncertainty.

I visually inspected the results for functions 1 and 2. For F1, although many observed outputs appear to be zero, the GP estimates a local maximum. For F2, the GP estimates several local maxima. In both plots, the suggested point appears close to a local maximum, although it does not exactly match the expected maximum.

To compare the predicted mean and uncertainty at each suggested point, I calculated `r = std / abs(mean)`. The results are:

- **F1:** 0.7085
- **F2:** 0.3264
- **F3:** 1.0485
- **F4:** 0.6164
- **F5:** 0.2905
- **F6:** 0.3321
- **F7:** 0.9808
- **F8:** 0.1154

For F1 and F2, the predicted mean magnitude is greater than the raw standard deviation, which is consistent with the suggested points appearing close to the GP's local maxima. F3 has the highest ratio, followed by F7, while F8 has the lowest. A ratio close to 1 means that the raw standard deviation and the absolute predicted mean are similar in magnitude. It does not mean that uncertainty contributes exactly 50% to UCB, because UCB multiplies the standard deviation by `kappa`. These ratios describe the suggested points only.

I have not yet determined how to check whether a suggested point is close to the maximum of the acquisition function for functions that cannot be inspected visually.

**Which functions were most challenging to query, and why? What additional information would have helped?**

Based on these ratios, I think F3 and F8 would be useful to analyze in more detail. Why do they have different ratios despite using the same `kappa`? One possible explanation is that F3 has fewer initial samples than F8. I should check the sample counts and input coverage before drawing a conclusion. If this explanation is correct, the functions may need different strategies: for F3, I could start by sampling more points evenly across the domain; for F8, I could focus on refining a local maximum.

**How do you plan to adjust your strategy in future rounds?**

I plan to identify which functions have higher uncertainty across the input space and reduce it by selecting points that prioritize exploration. For functions with lower uncertainty, after checking that the model is near a local maximum, I could focus more on exploitation and refine that maximum using a precision criterion that I still need to define. For functions that cannot be inspected visually, I also need to check whether the suggested point is close to the maximum UCB value.

## After a real evaluation
Choose the matching `function_id`, enter the actual returned scalar, and run once. Do not run this cell with an invented value. The corresponding candidate and optimizer are stored in `suggested_points` and `optimizers_by_function`.

In [8]:
# function_id = 1  # Select the function that was evaluated.
# observed_value = ...  # Replace only with the actual evaluator response.
# point = suggested_points[function_id]
# append_observation(root, function_id, point, observed_value, week=1)
# optimizers_by_function[function_id].observe(point, observed_value)
# print(
#     'Best observed:',
#     optimizers_by_function[function_id].history.best_input,
#     optimizers_by_function[function_id].history.best_output,
# )
